# 01 · Camera: what is ahead, which side, how far

**YOLO-World** is given the wearable's eight object names, so *door*, *rickshaw*
and *CNG* work without training. Direction comes from where the box sits in the
picture; distance is a pinhole **estimate** from typical object heights
(`config.OBJECT_HEIGHT_M`). The wearable measures distance with its laser
instead; here it is approximate and shown with `~`.

In [ ]:
import sys, pathlib
LAB = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
from va_lab import config
config.ensure_dirs()
print("lab folder:", LAB)

In [ ]:
from va_lab.vision import Detector, open_camera, grab
det = Detector("world")          # downloads yolov8s-worldv2.pt into models/ the first time
print("objects:", det.supported, "| device:", det.device)

In [ ]:
import cv2, matplotlib.pyplot as plt
cap = open_camera(); frame = grab(cap); cap.release()
dets = det.detect(frame)
for d in dets:
    print(f"{d.obj:9s} {d.score:.2f}  {d.angle_deg:+5.1f} deg  ~{d.distance_m:.1f} m{' (cut off: closer)' if d.clipped else ''}")
img = frame.copy()
for d in dets:
    x1, y1, x2, y2 = map(int, d.box)
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(img, f"{d.obj} ~{d.distance_m:.1f}m", (x1, max(12, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
plt.figure(figsize=(8, 6)); plt.imshow(img[..., ::-1]); plt.axis("off"); plt.show()

## What the assistant would say for this picture

In [ ]:
from va_lab import announce
from va_lab.speech import Speaker
spk = Speaker(bangla=False)
u = announce.describe(dets)
print([spk.caption(i) for i in u.items])
spk.say(u)

## Measure your webcam's field of view
Stand the laptop 1.00 m from a wall. Mark where the left and right edges of the
picture fall on the wall and measure the width W in metres.

In [ ]:
import math
W = 1.40   # metres, measured
hfov = 2 * math.degrees(math.atan(W / 2 / 1.00))
print(f"horizontal field of view = {hfov:.1f} deg  ->  set CAMERA_HFOV_DEG in va_lab/config.py")

## Speed: YOLO-World vs plain COCO YOLOv8n

In [ ]:
import time
coco = Detector("coco")
for name, d in (("world", det), ("coco", coco)):
    d.detect(frame)
    t = time.perf_counter()
    for _ in range(20): d.detect(frame)
    print(f"{name:6s} {(time.perf_counter() - t) / 20 * 1000:6.1f} ms per frame, knows {d.supported}")

## Live preview (press Q in the window to close)

In [ ]:
cap = open_camera()
while True:
    ok, frame = cap.read()
    for d in det.detect(frame):
        x1, y1, x2, y2 = map(int, d.box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{d.obj} {d.angle_deg:+.0f}deg ~{d.distance_m:.1f}m", (x1, max(12, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
    cv2.imshow("detections (Q to close)", frame)
    if cv2.waitKey(1) & 0xFF in (ord("q"), 27): break
cap.release(); cv2.destroyAllWindows()